In [ ]:
import re       # built-in: split and escape
import regex    # pip install regex  (re cannot read \p{L})

PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
compiled = regex.compile(PAT)

In [ ]:
def pretokenize(data, special_tokens):
    # 1. Cut data at the special tokens
    if special_tokens:
        toks = sorted(special_tokens, key=len, reverse=True)   # longest first
        pattern = "|".join(re.escape(t) for t in toks)         # escape makes | plain text
        chunks = re.split(pattern, data)
    else:
        chunks = [data]                                        # nothing to cut

    # 2. Run PAT on each chunk
    pre_token = []
    for chunk in chunks:
        pre_token.extend(m.group(0) for m in compiled.finditer(chunk))

    return chunks, pre_token

In [ ]:
# Test 1: one special token
data           = "hi<|endoftext|>ho"
special_tokens = ["<|endoftext|>"]

chunks, pre_token = pretokenize(data, special_tokens)
print("chunks   ", chunks)      # ['hi', 'ho']
print("pre_token", pre_token)   # ['hi', 'ho']

In [ ]:
# Test 2: two different special tokens
data           = "a cat<|endoftext|>a dog<|pad|>bye"
special_tokens = ["<|endoftext|>", "<|pad|>"]

chunks, pre_token = pretokenize(data, special_tokens)
print("chunks   ", chunks)      # ['a cat', 'a dog', 'bye']
print("pre_token", pre_token)   # ['a', ' cat', 'a', ' dog', 'bye']

In [ ]:
# Test 3: no special tokens
data           = "hi<|endoftext|>ho"
special_tokens = []

chunks, pre_token = pretokenize(data, special_tokens)
print("chunks   ", chunks)      # ['hi<|endoftext|>ho']
print("pre_token", pre_token)   # ['hi', '<|', 'endoftext', '|>', 'ho']

In [ ]:
# Test 4: token at the edges and back to back
# Predict chunks and pre_token first. Then run.
data           = "<|endoftext|>hi<|endoftext|><|endoftext|>ho"
special_tokens = ["<|endoftext|>"]

chunks, pre_token = pretokenize(data, special_tokens)
print("chunks   ", chunks)
print("pre_token", pre_token)

In [ ]:
# Bonus: predict first. Then run.
re.split("", "abc")